# 🔄 Lesson 4.5 — Transfer Learning and Fine-Tuning

**AI Crash Course for Absolute Beginners**

Why train from scratch when a pretrained model already knows edges, textures, and grammar? In this notebook:
- Load a pretrained ResNet-50 and use it as a feature extractor
- Fine-tune the last layers on a small custom dataset
- Compare: scratch vs feature extraction vs full fine-tuning
- Fine-tune a text classifier with Hugging Face

> Run each cell with **Shift + Enter**. GPU strongly recommended (free in Colab).

## ⚠️ GPU Strongly Recommended + Large Downloads

This notebook trains ResNet-50 and fine-tunes DistilBERT. On CPU this can take **30+ minutes**.

**Enable GPU first:**
1. **Runtime** menu → **Change runtime type**
2. Set **Hardware accelerator** to **T4 GPU** → **Save**
3. Estimated run time with GPU: ~8–12 minutes total

**Downloads this notebook makes:**
| Part | Download | Size |
|------|----------|------|
| Part 1 | ResNet-50 pretrained weights | ~100 MB |
| Part 4 | DistilBERT + SST-2 dataset | ~270 MB |

Each downloads once and is cached for the rest of the session.

In [ ]:
!pip install torch torchvision transformers datasets matplotlib --quiet

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models
# torchvision.models — contains pretrained models (ResNet, EfficientNet, VGG, etc.)
# These were trained on ImageNet (1.2 million images, 1000 classes)
# We can reuse their learned weights instead of training from scratch
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

---
## Part 1 — Load a Pretrained ResNet-50

In [ ]:
# ResNet-50 trained on ImageNet (1.2M images, 1000 classes)
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

total_params = sum(p.numel() for p in resnet.parameters())
print(f"ResNet-50 total parameters: {total_params:,}")
print(f"That's {total_params/1e6:.1f} million parameters!")
print("\nLast few layers:")
print(resnet.layer4[-1])
print(f"\nFinal classifier: {resnet.fc}")

In [ ]:
# Use it zero-shot on ImageNet classes
from PIL import Image
import urllib.request
import json

# Download ImageNet class labels
url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
urllib.request.urlretrieve(url, "imagenet_labels.json")
with open("imagenet_labels.json") as f:
    imagenet_labels = json.load(f)

# Download a sample image
img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg"
urllib.request.urlretrieve(img_url, "sample.jpg")

# Preprocess (ImageNet normalisation)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img = Image.open("sample.jpg").convert("RGB")
tensor = preprocess(img).unsqueeze(0)

resnet.eval()
with torch.no_grad():
    logits = resnet(tensor)
    probs  = torch.softmax(logits, dim=1)[0]
    top5   = torch.topk(probs, 5)

print("Top-5 ImageNet predictions:")
for score, idx in zip(top5.values, top5.indices):
    print(f"  {imagenet_labels[idx]:<25} {score:.3f}")

plt.imshow(img)
plt.axis("off")
plt.title(f"Predicted: {imagenet_labels[top5.indices[0]]}")
plt.show()

---
## Part 2 — Feature Extraction on CIFAR-10

In [ ]:
# Feature extraction: we FREEZE all of ResNet's weights
# "Freezing" means setting requires_grad=False — PyTorch will not calculate gradients for these layers
# and the optimizer will not update them
feature_extractor = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

for param in feature_extractor.parameters():
    param.requires_grad = False    # freeze all layers — their weights are locked

# ResNet-50 ends with a fully connected layer (fc) that predicts 1000 ImageNet classes
# We replace it with a new layer that predicts just 10 CIFAR-10 classes
# in_features is the number of inputs to the original final layer (2048 for ResNet-50)
in_features = feature_extractor.fc.in_features
feature_extractor.fc = nn.Linear(in_features, 10)   # ONLY this new layer has requires_grad=True
feature_extractor = feature_extractor.to(device)

trainable = sum(p.numel() for p in feature_extractor.parameters() if p.requires_grad)
total     = sum(p.numel() for p in feature_extractor.parameters())
print(f"Trainable parameters : {trainable:,}  ({trainable/total:.1%} of total)")
print(f"Frozen parameters    : {total-trainable:,}")

In [ ]:
# CIFAR-10 data (upscale to 224x224 for ResNet)
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Use only a subset for speed (2000 train, 500 test)
from torch.utils.data import Subset
full_train = datasets.CIFAR10(root="data", train=True,  download=True, transform=transform_train)
full_test  = datasets.CIFAR10(root="data", train=False, download=True, transform=transform_test)

train_subset = Subset(full_train, range(2000))
test_subset  = Subset(full_test,  range(500))

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_subset,  batch_size=64)

classes = full_train.classes
print(f"Classes: {classes}")

In [ ]:
def train_model(model, train_loader, test_loader, epochs=5, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    # filter(lambda p: p.requires_grad, model.parameters()) only passes UNFROZEN parameters
    # to the optimizer — frozen layers won't be updated even if accidentally included
    # lambda p: p.requires_grad is an anonymous function that returns True if a parameter is trainable
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    train_accs, test_accs = [], []

    for epoch in range(epochs):
        model.train()
        correct = total = 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            correct += (model(X).argmax(1) == y).sum().item()
            total   += len(y)
        train_accs.append(correct / total)

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                correct += (model(X).argmax(1) == y).sum().item()
                total   += len(y)
        test_accs.append(correct / total)
        print(f"  Epoch {epoch+1}/{epochs}  train={train_accs[-1]:.3f}  test={test_accs[-1]:.3f}")

    return train_accs, test_accs


print("Training feature extractor (only the new head):")
tr_fe, te_fe = train_model(feature_extractor, train_loader, test_loader, epochs=5)

---
## Part 3 — Full Fine-Tuning (Unfreeze Everything)

In [ ]:
fine_tuned = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
fine_tuned.fc = nn.Linear(fine_tuned.fc.in_features, 10)
fine_tuned = fine_tuned.to(device)

# Lower LR for backbone, higher for new head
backbone_params = [p for name, p in fine_tuned.named_parameters() if "fc" not in name]
head_params     = fine_tuned.fc.parameters()

print("Training with discriminative learning rates:")
print("  Backbone: lr=1e-5")
print("  New head: lr=1e-3")

tr_ft, te_ft = train_model(fine_tuned, train_loader, test_loader, epochs=5, lr=1e-4)

In [ ]:
# Compare results
plt.figure(figsize=(8, 4))
plt.plot(te_fe, label="Feature Extraction", marker="o", color="#1a6bc8")
plt.plot(te_ft, label="Full Fine-Tuning",   marker="s", color="#c8401a")
plt.xlabel("Epoch")
plt.ylabel("Test Accuracy")
plt.title("Feature Extraction vs Full Fine-Tuning on CIFAR-10 (2k samples)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Feature extraction final: {te_fe[-1]:.1%}")
print(f"Full fine-tuning final  : {te_ft[-1]:.1%}")
print("\nWith more data, the gap typically widens in favour of full fine-tuning.")

---
## Part 4 — Text: Fine-Tune DistilBERT for Sentiment

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from datasets import load_dataset
# Hugging Face `datasets` library — one-line access to thousands of NLP datasets
# AutoTokenizer automatically picks the right tokenizer for the model
# AutoModelForSequenceClassification loads a transformer with a classification head on top

# Load a small subset of SST-2 (Stanford Sentiment Treebank — movie reviews)
# "train[:1000]" means only take the first 1000 training examples (fast for demo)
dataset = load_dataset("glue", "sst2", split={"train": "train[:1000]", "validation": "validation[:200]"})

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    # truncation=True cuts sentences longer than max_length to fit the model
    # padding="max_length" pads shorter sentences with zeros to make all inputs the same length
    # max_length=64 — we use short sequences for speed (BERT can handle up to 512)
    return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=64)

# batched=True processes multiple examples at once (much faster than one at a time)
dataset = dataset.map(tokenize, batched=True)
# The Trainer expects the label column to be named "labels"
dataset = dataset.rename_column("label", "labels")
# set_format("torch") converts the dataset to PyTorch tensors automatically
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Training examples  : {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")
print(f"Sample: '{dataset['train'][0]}'")

In [ ]:
from sklearn.metrics import accuracy_score

# AutoModelForSequenceClassification.from_pretrained loads DistilBERT's pretrained weights
# num_labels=2 adds a 2-class classification head on top (positive/negative)
model_bert = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

# TrainingArguments controls all training settings in one place
args = TrainingArguments(
    output_dir="./results",            # where to save model checkpoints
    num_train_epochs=3,                # how many full passes through the training data
    per_device_train_batch_size=32,    # batch size per GPU (or CPU if no GPU)
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",       # evaluate on the validation set after each epoch
    logging_steps=50,                  # print training loss every 50 steps
    save_strategy="no",                # don't save checkpoints (saves disk space for this demo)
    report_to="none"                   # don't log to Weights & Biases or other tracking tools
)

def compute_metrics(eval_pred):
    # eval_pred is a tuple of (logits, labels) passed in automatically by the Trainer
    logits, labels = eval_pred
    # np.argmax(logits, axis=1) picks the class with the highest score for each example
    # axis=1 means "take the max across columns" (each row = one example, columns = classes)
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

trainer = Trainer(
    model=model_bert,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(f"\nFine-tuned DistilBERT accuracy: {results['eval_accuracy']:.1%}")
print("(Trained on only 1,000 examples for 3 epochs)")

---
## When to Use Which Strategy?

| Scenario | Approach | Typical LR |
|---|---|---|
| Small dataset (<1k), similar domain | Feature extraction | 1e-3 on head only |
| Medium dataset (1k-10k) | Fine-tune last few layers | 1e-4 to 1e-5 |
| Large dataset (>10k) | Full fine-tuning | 1e-5 (backbone), 1e-4 (head) |
| Very different domain | Fine-tune all + more epochs | Lower LR, more epochs |

---
## Challenge Exercises

1. **Progressive unfreezing**: Start with only the head, train 2 epochs, then unfreeze layer4, train 2 more, then unfreeze all.
2. **Different backbone**: Try `models.efficientnet_b0()` instead of ResNet-50. Compare accuracy and parameter count.
3. **Your own images**: Use `torchvision.datasets.ImageFolder` to train on a custom folder of images.

---
**Next phase:** [Lesson 5.1 — AI Ethics](https://GirlEf.github.io/ai-crash-course/PHASE%20FIVE/LESSON%205.1.html)